In [25]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from xgboost import XGBClassifier

# Ignoro i warning 
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

# Percorso da cui prendere il file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Serve per far paritre il debug
DEBUG = False

# Carico il csv e stampo la shape

In [26]:
df_duke = pd.read_csv(FILE_PATH / "duke_lesions.csv")

print("DUKE shape: ", df_duke.shape)

DUKE shape:  (291, 109)


# Target

In [27]:
marker = ["PR", "ER"]

# Bilancio il dataset globalmente

In [28]:
def balance_training_set(X_train, y_train, target, random_state=42):
    df_train = pd.concat([X_train, y_train], axis=1)

    df_0 = df_train[df_train[target] == 0]
    df_1 = df_train[df_train[target] == 1]

    n_min = min(len(df_0), len(df_1))

    df_bal = pd.concat([
        df_0.sample(n=n_min, random_state=random_state),
        df_1.sample(n=n_min, random_state=random_state)
    ]).sample(frac=1, random_state=random_state)

    return df_bal[X_train.columns], df_bal[target]


# Controllo il numero delle classi

In [29]:
for m in marker:
    vc = df_duke[m].value_counts(dropna=False)

    print(f"\nDistribuzione {m} – DUKE")
    print("-" * 30)
    print("Negativi:", vc.get(0, 0))
    print("Positivi:", vc.get(1, 0))



Distribuzione PR – DUKE
------------------------------
Negativi: 157
Positivi: 134

Distribuzione ER – DUKE
------------------------------
Negativi: 123
Positivi: 168


# Stratified Cross-Validation

In [30]:
FEATURES = [c for c in df_duke.columns if c.startswith("original_")]

skf = StratifiedKFold(
    n_splits=5,
    random_state=42,
    shuffle=True
    
)

# Training

In [31]:
results = {}

for target in marker:
    print(f"\n==============================")
    print(f" Training e CV per {target}")
    print(f"==============================")

    X = df_duke[FEATURES]
    y = df_duke[target]


    acc_scores = []
    f1_scores = []
    auc_scores = []
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):

        # Split
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Bilanciamento solo in fase di  training
        # undersampling applicato solo al training set per evitare data leakage
        X_train_bal, y_train_bal = balance_training_set(X_train, y_train, target)

        if DEBUG:
            print(f"\n[DEBUG] {target} – Fold {fold}")
            print("X_train_bal shape:", X_train_bal.shape)
            print("y_train_bal shape:", y_train_bal.shape)

            print("Distribuzione classi bilanciate: ", y_train_bal.value_counts())


        # Definisco il modello
        model = XGBClassifier(
            random_state=42,
            n_jobs=-1,
            objective='binary:logistic',
            eval_metric='logloss',
            tree_method='hist',
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=3,
            reg_alpha=0,
            reg_lambda=1
        )

        model.fit(X_train_bal, y_train_bal)

        # Predizioni
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        # Metriche
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_prob)

        acc_scores.append(acc)
        f1_scores.append(f1)
        auc_scores.append(auc)
        
    if DEBUG:
        print(
            f"Fold {fold} | "
            f"ACC={acc:.3f} | "
            f"F1={f1:.3f} | "
            f"AUC={auc:.3f}"
        )

    # Media ± std
    results[target] = {
        "Accuracy": (np.mean(acc_scores), np.std(acc_scores)),
        "F1": (np.mean(f1_scores), np.std(f1_scores)),
        "AUC": (np.mean(auc_scores), np.std(auc_scores)),
    }

    print("\n--- RISULTATI FINALI ---")
    print(f"Accuracy : {results[target]['Accuracy'][0]:.3f} ± {results[target]['Accuracy'][1]:.3f}")
    print(f"F1-score : {results[target]['F1'][0]:.3f} ± {results[target]['F1'][1]:.3f}")
    print(f"ROC-AUC  : {results[target]['AUC'][0]:.3f} ± {results[target]['AUC'][1]:.3f}")






 Training e CV per PR

--- RISULTATI FINALI ---
Accuracy : 0.536 ± 0.074
F1-score : 0.512 ± 0.082
ROC-AUC  : 0.503 ± 0.083

 Training e CV per ER

--- RISULTATI FINALI ---
Accuracy : 0.578 ± 0.058
F1-score : 0.607 ± 0.058
ROC-AUC  : 0.559 ± 0.045
